# Lezione 11 — Costruendo un chatbot (parte 1: il database)

**Obiettivi di oggi**
- Capire come è strutturata una conversazione padre-risposta.
- Filtrare i commenti reali secondo i criteri di qualità.
- Costruire il database che alimenterà il chatbot.

Consulta il [capitolo 9 del manuale](../manual/09_chatbot.md) per la spiegazione completa.

**Nota**: en lugar de descargar el volcado completo de comentarios de Reddit (varios gigabytes), usamos aquí una pequeña muestra de ejemplo con la misma estructura, para practicar el pipeline completo de limpieza e inserción.

## 1. Un campione di commenti grezzi

Ogni riga simula un commento di Reddit: il suo identificatore, l'identificatore del genitore, il testo del genitore, il testo del commento, il subreddit, la data e il suo punteggio.

In [ ]:
comentarios_en_bruto = [
    {"comment_id": "c1", "parent_id": "p1", "parent": "¿Cuál es la mejor forma de aprender TensorFlow?", "comment": "Empieza por los fundamentos y practica con notebooks pequeños.", "subreddit": "MachineLearning", "score": 12},
    {"comment_id": "c2", "parent_id": "p2", "parent": "¿Qué opináis de este curso?", "comment": "[deleted]", "subreddit": "MachineLearning", "score": 1},
    {"comment_id": "c3", "parent_id": "p3", "parent": "¿Alguien ha entrenado una CNN 3D?", "comment": "", "subreddit": "MachineLearning", "score": 0},
    {"comment_id": "c4", "parent_id": "p4", "parent": "¿RNN o Transformer para este caso?", "comment": "Depende del tamaño del dataset, pero para secuencias cortas una RNN suele bastar.", "subreddit": "MachineLearning", "score": 8},
]

print(f"Comentarios en bruto: {len(comentarios_en_bruto)}")

## 2. Pulisci e convalida ogni commento

In [ ]:
def limpiar_texto(texto):
    texto = texto.replace("\n", " ").replace("\r", " ")
    texto = texto.replace('"', "'")
    return texto.strip()

def es_aceptable(comentario):
    if len(comentario) < 1 or len(comentario) > 1000:
        return False
    if comentario in ("[deleted]", "[removed]"):
        return False
    return True

comentarios_validos = []
for c in comentarios_en_bruto:
    texto_limpio = limpiar_texto(c["comment"])
    if es_aceptable(texto_limpio):
        c["comment"] = texto_limpio
        comentarios_validos.append(c)

print(f"Comentarios válidos tras filtrar: {len(comentarios_validos)} de {len(comentarios_en_bruto)}")

## 3. Crea il database SQLite con la tabella `parent_reply`


In [ ]:
import sqlite3

conexion = sqlite3.connect("chatbot.db")
cursor = conexion.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS parent_reply (
    parent_id TEXT,
    comment_id TEXT,
    parent TEXT,
    comment TEXT,
    subreddit TEXT,
    score INTEGER
)
""")
conexion.commit()

## 4. Inserisci i commenti validi


In [ ]:
for c in comentarios_validos:
    cursor.execute(
        "INSERT INTO parent_reply (parent_id, comment_id, parent, comment, subreddit, score) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        (c["parent_id"], c["comment_id"], c["parent"], c["comment"], c["subreddit"], c["score"]),
    )
conexion.commit()

cursor.execute("SELECT parent, comment FROM parent_reply")
for fila in cursor.fetchall():
    print("P:", fila[0])
    print("R:", fila[1])
    print()

## Riepilogo

- Filtraste comentarios vacíos, eliminados o demasiado largos antes de guardarlos.
- Construiste una base de datos SQLite con pares pregunta-respuesta, lista para convertirse en datos de entrenamiento.
- Con datos reales, este mismo proceso se aplica a millones de filas en lotes, no una a una.

Continua con il [notebook de práctica di questa lezione](../notebooks_practica/clase11_practica_chatbot_datos.ipynb).